In [1]:
import datetime
import numpy as np
import pyaurorax

aurorax = pyaurorax.PyAuroraX()
at = aurorax.tools
at.set_theme("dark")

# Plot satellite footprints on an ASI movie

This notebook overlays TRACERS-1, TRACERS-2, and REAL spacecraft footprints onto a TREx RGB movie at GILL for 2026-03-09 03:24–03:30 UT. The pattern extends to other satellites and ASI instruments.

In [2]:
# ASI station + dataset
site_uid = "gill"
dataset_name = "TREX_RGB_RAW_NOMINAL"
skymap_dataset = "TREX_RGB_SKYMAP_IDLSAV"

# Time window for both the ephemeris search and the ASI download. This gives the *overall* range of timestamps
# for the full ASI movie.
start_dt = datetime.datetime(2026, 3, 9, 3, 24, 0)
end_dt = datetime.datetime(2026, 3, 9, 3, 30, 0)

# Assumed altitude for the emission -- since we are using TREx-RGB, primarily dominated by the 557.7 nm green-line
# emission, we choose a typical height of 110 km
altitude_km = 110

# Define a dictionary to include some information about each spacecraft platform we want to include. In this example, we plot
# three successive overflights of TRACERS 1, TRACERS 2, and REAL. Here, we also define the colour to plot each satellite.
#
# NOTE: We set individual start and end times for each spacecraft. This is optional, to restrict plotting of each footprint to
# times when the footprint is actually within the FoV of the ASI. This allows footprints to successively appear and dissapear
# as they enter and leave the ASI FoV.
satellites = {
    "tracers1": {
        "color":  "blue",
        "window": (datetime.datetime(2026, 3, 9, 3, 24, 0),
                   datetime.datetime(2026, 3, 9, 3, 29, 45)),
    },
    "tracers2": {
        "color":  "red",
        "window": (datetime.datetime(2026, 3, 9, 3, 24, 0),
                   datetime.datetime(2026, 3, 9, 3, 29, 12)),
    },
    "real": {
        "color":  "magenta",
        "window": (datetime.datetime(2026, 3, 9, 3, 25, 51),
                   datetime.datetime(2026, 3, 9, 3, 27, 48)),
    },
}

# Set the scaling bounds for the ASI data
intensity_min = 10
intensity_max = 120

# Define how long of a trail is left behind the footprint plot point (visual effect only)
trail_seconds = 30

# NOTE: This should match the cadence of whatever ASI data we are plotting on top of. In this case, we
# use 3-seconds for nominal TREx-RGB data. The AuroraX ephemeris search obtains spacecraft footprints
# from SSCWeb, which provides locations at a cadence of 1-minute. Thus, we will interpolate the footprints
# to match the cadence of our data.
ephemeris_cadence_sec = 3

Next, we will use `aurorax.search.ephemeris.search` to obtain the satellite footprints over the time of interest

In [3]:
programs  = ["tracers", "real"]
platforms = list(satellites.keys())

s = aurorax.search.ephemeris.search(
    start_dt,
    end_dt,
    programs=programs,
    platforms=platforms,
    instrument_types=["footprint"],
    verbose=True,
)
print(f"Got {len(s.data)} ephemeris records.")

[2026-05-10 19:45:57.921915] Search object created
[2026-05-10 19:45:57.965137] Request submitted
[2026-05-10 19:45:57.965175] Request ID: 2fa00c0f-fa50-48e9-881a-c8772be7551a
[2026-05-10 19:45:57.965181] Request details available at: https://api.aurorax.space/api/v1/ephemeris/requests/2fa00c0f-fa50-48e9-881a-c8772be7551a
[2026-05-10 19:45:57.965186] Waiting for data ...
[2026-05-10 19:45:59.838700] Checking for data ...
[2026-05-10 19:46:00.732484] Data is now available
[2026-05-10 19:46:00.732595] Retrieving data ...
[2026-05-10 19:46:00.844833] Retrieved 91.9 kB of data containing 21 records
Got 21 ephemeris records.


Split the ephemeris results by platform and interpolate to the ASI cadence.

In [4]:
def split_and_interp(records, platform_name, target_cadence_sec):
    """Split ephemeris results by platform and linearly interpolate to target cadence.
    Returns (datetime_list, lat_array, lon_array), or None if the platform has no records.
    """
    times, lats, lons = [], [], []
    for r in records:
        if r.data_source.platform != platform_name:
            continue
        epoch = datetime.datetime.fromisoformat(r.epoch) if isinstance(r.epoch, str) else r.epoch
        times.append(epoch)
        lats.append(r.location_geo.lat)
        lons.append(r.location_geo.lon)

    if not times:
        return None

    lats = np.asarray(lats)
    lons = np.asarray(lons)

    # Build the fine-cadence time grid spanning the original window.
    t0 = times[0]
    duration_sec = (times[-1] - t0).total_seconds()
    n_steps = int(duration_sec / target_cadence_sec) + 1
    fine_times = [t0 + datetime.timedelta(seconds=i * target_cadence_sec) for i in range(n_steps)]

    # Interpolate in seconds (since t0)
    coarse_t_sec = np.array([(t - t0).total_seconds() for t in times])
    fine_t_sec   = np.array([(t - t0).total_seconds() for t in fine_times])
    fine_lats = np.interp(fine_t_sec, coarse_t_sec, lats)
    fine_lons = np.interp(fine_t_sec, coarse_t_sec, lons)

    return fine_times, fine_lats, fine_lons

In [5]:
# Use the above helper function to split and interpolate the per-satellite records
interp_ephemeris = {}
for platform_name in satellites:
    result = split_and_interp(s.data, platform_name, target_cadence_sec=ephemeris_cadence_sec)
    if result is None:
        print(f"WARNING: no ephemeris records returned for {platform_name}")
        continue
    interp_ephemeris[platform_name] = result
    print(f"{platform_name:8s}  {len(result[0])} interpolated points  "
          f"({result[0][0]} -> {result[0][-1]})")

tracers1  121 interpolated points  (2026-03-09 03:24:00 -> 2026-03-09 03:30:00)
tracers2  121 interpolated points  (2026-03-09 03:24:00 -> 2026-03-09 03:30:00)
real      121 interpolated points  (2026-03-09 03:24:00 -> 2026-03-09 03:30:00)


Download and scale the ASI data.

In [6]:
r = aurorax.data.ucalgary.download(dataset_name, start_dt, end_dt, site_uid=site_uid)
data = aurorax.data.ucalgary.read(r.dataset, r.filenames, n_parallel=5)
print(f"data.data shape: {data.data.shape}    n_frames: {data.data.shape[-1]}")
print(f"first timestamp: {data.timestamp[0]}")
print(f"last timestamp:  {data.timestamp[-1]}")

data.data shape: (480, 553, 3, 140)    n_frames: 140
first timestamp: 2026-03-09 03:24:00.283407
last timestamp:  2026-03-09 03:30:57.246674


In [7]:
images_scaled = at.scale_intensity(data.data, min=intensity_min, max=intensity_max)
print(f"images_scaled shape: {images_scaled.shape}")

images_scaled shape: (480, 553, 3, 140)


Download the skymap. We need it to map satellite lat/lon to CCD pixels at the assumed emission altitude.

In [8]:
mid_dt = start_dt + (end_dt - start_dt) / 2
skymap_download = aurorax.data.ucalgary.download_best_skymap(skymap_dataset, site_uid, mid_dt)
skymap_data = aurorax.data.ucalgary.read(skymap_download.dataset, skymap_download.filenames)
skymap = skymap_data.data[0]
print(f"skymap altitudes available: {skymap.full_map_altitude / 1000.0} km")

skymap altitudes available: [ 90. 110. 150.] km


Generate one PNG per frame, overlaying each visible satellite's trail and location marker.

In [9]:
import os
import platform
import matplotlib.pyplot as plt
from tqdm.contrib.concurrent import process_map as tqdm_process_map
from tqdm import tqdm

# Convert trail length from seconds to samples on the interpolated ephemeris grid.
n_trail = int(round(trail_seconds / ephemeris_cadence_sec))

def process_frame(i):
    _, ax = at.display(images_scaled[:, :, :, i], returnfig=True)
    frame_dt = data.timestamp[i]

    for platform_name, info in satellites.items():
        win_start, win_end = info["window"]
        if not (win_start <= frame_dt <= win_end):
            continue
        if platform_name not in interp_ephemeris:
            continue

        sat_times, sat_lats, sat_lons = interp_ephemeris[platform_name]
        deltas = np.array([(t - frame_dt).total_seconds() for t in sat_times])
        i_now = int(np.argmin(np.abs(deltas)))
        i_tail_start = max(0, i_now - n_trail + 1)
        tail_lats = sat_lats[i_tail_start:i_now + 1]
        tail_lons = sat_lons[i_tail_start:i_now + 1]

        # Project the satellite's geographic trail onto CCD pixels via the skymap.
        try:
            x_pix, y_pix = at.ccd_contour.geo(
                skymap,
                altitude_km=altitude_km,
                contour_lats=tail_lats,
                contour_lons=tail_lons,
                remove_edge_cases=True,
            )
        except Exception:
            continue
        if len(x_pix) == 0:
            continue

        ax.plot(x_pix, y_pix, color=info["color"], linewidth=1.5, alpha=0.7)
        ax.plot(x_pix[-1], y_pix[-1], marker="o", color=info["color"], markersize=8)
        ax.text(x_pix[-1] - 6, y_pix[-1], platform_name.upper(),
                color=info["color"], fontsize=11, ha="right", va="center")

    ax.text(5, 535, "TREx RGB",                        color="white", size=14)
    ax.text(5, 510, site_uid.upper(),                  color="white", size=14)
    ax.text(5, 25,  frame_dt.strftime("%Y-%m-%d"),     color="white", size=11)
    ax.text(5, 10,  frame_dt.strftime("%H:%M:%S UTC"), color="white", size=11)

    filename = "movie_frames/%s_%s_trex_rgb_footprint.png" % (
        data.timestamp[i].strftime("%Y%m%d_%H%M%S"), site_uid)
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    plt.savefig(filename)
    plt.close()
    return filename


if (platform.system() == "Windows"):
    # NOTE: multiprocessing on Windows from within a notebook is not too easy, so we'll
    # just do this serially.
    frame_filename_list = []
    for i in tqdm(range(0, images_scaled.shape[-1]), total=images_scaled.shape[-1],
                  desc="Generating frame files: ", unit="frames"):
        frame_filename_list.append(process_frame(i))
else:
    frame_filename_list = tqdm_process_map(
        process_frame,
        range(0, images_scaled.shape[-1]),
        max_workers=5,
        chunksize=1,
        desc="Generating frame files: ",
        unit="frames",
    )

Generating frame files:   0%|          | 0/140 [00:00<?, ?frames/s]

Combine the frames into a movie using the AuroraX movie function.

In [10]:
at.movie(frame_filename_list, "test_trex_rgb_footprint_movie.mp4", n_parallel=5, fps=15)

Reading files:   0%|          | 0/140 [00:00<?, ?files/s]

Encoding frames:   0%|          | 0/140 [00:00<?, ?frames/s]